In [3]:
import pandas as pd
import os

def merge_crypto_data():
    """Merge historical crypto data with market metadata - ALL COLUMNS"""
    
    print("=" * 70)
    print("MERGING CRYPTO DATA (ALL COLUMNS)")
    print("=" * 70)
    
    # File paths
    history_path = '../outputs/crypto_history.csv'
    market_path = '../outputs/crypto_market.csv'
    output_path = '../outputs/crypto_data.csv'
    
    # 1. Check if files exist
    print("\n[1/4] Checking input files...")
    if not os.path.exists(history_path):
        print(f"❌ Error: {history_path} not found!")
        return
    if not os.path.exists(market_path):
        print(f"❌ Error: {market_path} not found!")
        return
    print("✓ Both files found")
    
    # 2. Load data
    print("\n[2/4] Loading data...")
    history = pd.read_csv(history_path)
    market = pd.read_csv(market_path)
    
    print(f"✓ History data: {len(history):,} rows, {len(history.columns)} columns")
    print(f"✓ Market data: {len(market):,} rows, {len(market.columns)} columns")
    print(f"✓ Unique symbols in history: {history['symbol'].nunique():,}")
    print(f"✓ Unique symbols in market: {market['symbol'].nunique():,}")
    
    # 3. Check for overlapping columns (except 'symbol')
    print("\n[3/4] Analyzing column overlap...")
    history_cols = set(history.columns)
    market_cols = set(market.columns)
    
    overlapping = history_cols.intersection(market_cols) - {'symbol'}
    
    if overlapping:
        print(f"⚠️  Warning: {len(overlapping)} overlapping columns detected:")
        print(f"   {sorted(overlapping)}")
        print(f"   These will get '_history' and '_market' suffixes")
    else:
        print("✓ No overlapping columns (except 'symbol')")
    
    print(f"\nColumns from history: {len(history.columns)}")
    print(f"Columns from market: {len(market.columns)}")
    
    # 4. Merge ALL columns from both dataframes
    print("\n[4/4] Merging ALL columns...")
    df = history.merge(
        market, 
        on='symbol', 
        how='left',
        suffixes=('_history', '_market')  # Handle duplicate columns
    )
    
    # Check for unmatched symbols
    # Use the first column from market (excluding symbol) to check for matches
    market_indicator_col = [col for col in market.columns if col != 'symbol'][0]
    if market_indicator_col in df.columns:
        unmatched = df[df[market_indicator_col].isna()]['symbol'].nunique()
    else:
        # Check with suffix
        market_indicator_col_suffixed = market_indicator_col + '_market'
        if market_indicator_col_suffixed in df.columns:
            unmatched = df[df[market_indicator_col_suffixed].isna()]['symbol'].nunique()
        else:
            unmatched = 0
    
    print(f"✓ Merged data: {len(df):,} rows, {len(df.columns)} columns")
    if unmatched > 0:
        print(f"⚠️  Warning: {unmatched} symbols from history had no match in market data")
    
    # 5. Save merged data
    print("\n[5/5] Saving merged data...")
    df.to_csv(output_path, index=False)
    print(f"✓ Saved to: {output_path}")
    
    # Summary statistics
    print("\n" + "=" * 70)
    print("MERGE SUMMARY")
    print("=" * 70)
    print(f"Input rows (history): {len(history):,}")
    print(f"Output rows: {len(df):,}")
    print(f"Columns from history: {len(history.columns)}")
    print(f"Columns from market: {len(market.columns)}")
    print(f"Total columns in output: {len(df.columns)}")
    print(f"Columns added: {len(df.columns) - len(history.columns)}")
    
    # Show all column names
    print("\n" + "=" * 70)
    print("ALL COLUMNS IN MERGED DATA")
    print("=" * 70)
    print(f"\nTotal: {len(df.columns)} columns")
    print("\nColumn names:")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i:3d}. {col}")
    
    if 'date' in df.columns or 'timestamp' in df.columns:
        date_col = 'date' if 'date' in df.columns else 'timestamp'
        print(f"\nDate range: {df[date_col].min()} to {df[date_col].max()}")
    
    if 'market_cap_category' in df.columns:
        print("\nMarket cap distribution:")
        print(df['market_cap_category'].value_counts().to_string())
    
    # Data preview
    print("\n" + "=" * 70)
    print("DATA PREVIEW (First 3 rows, first 10 columns)")
    print("=" * 70)
    print(df.head(3).iloc[:, :10].to_string())
    
    print("\n" + "=" * 70)
    print("DATA INFO")
    print("=" * 70)
    print(df.info())
    
    print("\n✅ Merge complete!")
    return df


if __name__ == "__main__":
    merged_df = merge_crypto_data()

MERGING CRYPTO DATA (ALL COLUMNS)

[1/4] Checking input files...
✓ Both files found

[2/4] Loading data...
✓ History data: 5,772,244 rows, 4 columns
✓ Market data: 4,772 rows, 28 columns
✓ Unique symbols in history: 4,741
✓ Unique symbols in market: 4,772

[3/4] Analyzing column overlap...
⚠️  Warning: 2 overlapping columns detected:
   ['price', 'volume']
   These will get '_history' and '_market' suffixes

Columns from history: 4
Columns from market: 28

[4/4] Merging ALL columns...
✓ Merged data: 5,772,244 rows, 31 columns

[5/5] Saving merged data...


In [1]:
import pandas as pd

df = pd.read_csv('../outputs/crypto_market.csv')
list(set(df['market_cap_category']))


['Mid (100M-1B)',
 'Micro (<10M)',
 'Small (10M-100M)',
 'Mega (>10B)',
 'Large (1B-10B)',
 nan]

In [2]:
df['market_cap_category'].value_counts()

market_cap_category
Micro (<10M)        2349
Small (10M-100M)     385
Mid (100M-1B)        161
Large (1B-10B)        48
Mega (>10B)           17
Name: count, dtype: int64

In [8]:
market[market['market_cap_category'] == '2026-02-09 11:30:31']

,symbol,name,exchange,icoDate,circulatingSupply,totalSupply,price,changePercentage,change,volume,...,open,previousClose,timestamp,daily_range_pct,ytd_gain_pct,distance_from_50ma_pct,distance_from_200ma_pct,market_cap_category,volume_to_mcap_ratio,data_collected_at
2764,MIBUSD,MIB Coin USD,CRYPTO,2018-09-04,0.0,300000000.0,0.000035,0.0,0.0,18.0,...,0.000035,0.000035,1744894140,0.0,0.49,-76.66,-83.99,2026-02-09 11:30:31,NaN,NaN


CRYPTOCURRENCY DATA COLLECTION (Parallel Mode)

[1/4] Fetching cryptocurrency list...
✓ Fetched 4785 cryptocurrencies

[2/4] Checking for existing data...
📋 Need to fetch: 4785 symbols
✓ Already have: 0 symbols
🎯 Total: 4785 symbols

[3/4] Fetching data with 20 parallel workers...

Progress: 50/4785 (1.0%) | Success: 49 | Failed: 0 | Rate: 19.4/s | ETA: 0:04:04
Progress: 100/4785 (2.1%) | Success: 99 | Failed: 0 | Rate: 21.8/s | ETA: 0:03:34
Progress: 150/4785 (3.1%) | Success: 149 | Failed: 0 | Rate: 22.1/s | ETA: 0:03:29
Progress: 200/4785 (4.2%) | Success: 199 | Failed: 0 | Rate: 22.7/s | ETA: 0:03:22
Progress: 250/4785 (5.2%) | Success: 249 | Failed: 0 | Rate: 22.9/s | ETA: 0:03:18
Progress: 300/4785 (6.3%) | Success: 299 | Failed: 0 | Rate: 23.1/s | ETA: 0:03:14
Progress: 350/4785 (7.3%) | Success: 349 | Failed: 0 | Rate: 23.0/s | ETA: 0:03:13
Progress: 400/4785 (8.4%) | Success: 399 | Failed: 0 | Rate: 23.1/s | ETA: 0:03:09
Progress: 450/4785 (9.4%) | Success: 449 | Failed: 0 | R